# 01 — Data Collection

End-to-end pull from Steam Storefront, Steam Web API, SteamSpy, IsThereAnyDeal, and SteamCharts into `data/steam.db`.

**Resumable.** Each stage updates progress flags on the `app_list` table, so re-running a stage only fetches the appids that haven't been done yet. Safe to interrupt and restart.

**Order matters:** sample pool → Steam details (filters out non-games) → reviews → SteamSpy → ITAD mappings → ITAD price history → SteamCharts historical CCU. SteamCharts must run *after* details, since `steamcharts_history` has a FK to `games`.

**Time budget:** with 5,000 games and conservative throttles, expect Storefront ~2–3h, ITAD ~1h, SteamCharts ~3h. Run overnight or in chunks via the `LIMIT` in each stage.

## Setup

In [1]:
import sys
from pathlib import Path

# Make `src` importable when the notebook is launched from `notebooks/`.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import db, steam_api, steamspy_api, itad_api, steamcharts
from src.utils import load_keys

keys = load_keys()
assert keys['steam'], 'STEAM_API_KEY missing from .env'
assert keys['itad'], 'ISTHEREANYDEAL_API missing from .env'
print('keys loaded')

keys loaded


In [3]:
conn = db.connect()
db.init_db(conn)

# Quick schema check
tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
print('tables:', tables)

tables: ['app_list', 'game_categories', 'game_genres', 'games', 'itad_mapping', 'player_counts', 'price_history', 'review_timestamps', 'reviews_summary', 'steamcharts_history', 'steamspy', 'steamspy_tags']


## Stage 1 — Build the sample pool

Pull top-owned games from SteamSpy (5 pages × 1000 = ~5,000 candidates) and seed them into `app_list`. SteamSpy's `request=all` is rate-limited to 1 call per 60s, so this stage takes ~5 min.

In [3]:
pool = steamspy_api.fetch_sample_pool(num_pages=5)
print(f'pulled {len(pool)} candidate games from SteamSpy')

inserted = db.add_to_app_list(
    conn,
    [(p['appid'], p['name']) for p in pool],
    source='steamspy_top',
)
print(f'{inserted} new appids added to app_list')
print('app_list size:', conn.execute('SELECT COUNT(*) FROM app_list').fetchone()[0])

pulled 5000 candidate games from SteamSpy
4995 new appids added to app_list
app_list size: 4995


## Stage 2 — Steam Storefront details

The bottleneck stage. Fetches `/api/appdetails` for each appid (cc=ph for PHP pricing) and decomposes the response into `games`, `game_genres`, `game_categories`. Non-`game` types (DLC, demo, soundtrack) get their flag set to 1 with `last_error='type=...'` so we don't retry them.

Run this cell repeatedly with a `LIMIT` if you want to checkpoint progress.

In [4]:
# Take a small batch first to verify the pipeline end-to-end before going big.
batch = db.pending_appids(conn, 'has_details', limit=20)
print(f'fetching details for {len(batch)} games...')
stats = steam_api.collect_app_details(conn, batch, cc='ph')
print(stats)

fetching details for 20 games...
{'ok': 20, 'missing': 0, 'skipped': 0, 'error': 0}


In [5]:
# Full run — uncomment when ready. ~1.5s per request, so 5000 games ≈ 2h.
batch = db.pending_appids(conn, 'has_details')
stats = steam_api.collect_app_details(conn, batch, cc='ph')
print(stats)

{'ok': 4903, 'missing': 46, 'skipped': 3, 'error': 0}


In [6]:
# Sanity check — what landed in `games`?
import pandas as pd
pd.read_sql('SELECT appid, title, type, currency, launch_price_cents, current_price_cents, discount_percent, release_date FROM games LIMIT 10', conn)

,appid,title,type,currency,launch_price_cents,current_price_cents,discount_percent,release_date
0,10,Counter-Strike,game,PHP,28995,28995,0,"1 Nov, 2000"
1,20,Team Fortress Classic,game,PHP,17495,17495,0,"1 Apr, 1999"
2,30,Day of Defeat,game,PHP,17495,17495,0,"1 May, 2003"
3,40,Deathmatch Classic,game,PHP,17495,17495,0,"1 Jun, 2001"
4,50,Half-Life: Opposing Force,game,PHP,17495,17495,0,"1 Nov, 1999"
5,60,Ricochet,game,PHP,17495,17495,0,"1 Nov, 2000"
6,70,Half-Life,game,PHP,28995,28995,0,"19 Nov, 1998"
7,80,Counter-Strike: Condition Zero,game,PHP,28995,28995,0,"1 Mar, 2004"
8,100,Counter-Strike: Condition Zero,game,PHP,28995,28995,0,"1 Mar, 2004"
9,130,Half-Life: Blue Shift,game,PHP,17495,17495,0,"1 Jun, 2001"


## Stage 3 — Review summaries

Aggregate review counts + score per game. Set `fetch_individual=True` to also pull paginated review timestamps (used for review-velocity in Part 2). Pulling individual reviews multiplies the request count, so leave it off for the first pass.

In [ ]:
batch = db.pending_appids_with_game(conn, 'has_reviews', limit=50)
stats = steam_api.collect_reviews(conn, batch, fetch_individual=False)
print(stats)

In [ ]:
# When you're ready for review timestamps (Part 2):
# batch = db.pending_appids_with_game(conn, 'has_reviews')
# steam_api.collect_reviews(conn, batch, fetch_individual=True, max_pages=5)

## Stage 4 — SteamSpy per-app details

Owners estimates, playtime medians, and community tags. Throttled at ~1/sec — 5000 games ≈ 90 min.

In [ ]:
batch = db.pending_appids_with_game(conn, 'has_steamspy', limit=50)
stats = steamspy_api.collect_steamspy(conn, batch)
print(stats)

## Stage 5 — IsThereAnyDeal

Two sub-stages: first translate Steam appid → ITAD UUID, then pull historical price points (Steam shop only — `shop_id=61`) for the games we successfully mapped. ITAD coverage may miss obscure indies.

In [ ]:
batch = db.pending_appids_with_game(conn, 'has_itad_id', limit=50)
stats = itad_api.collect_itad_mappings(conn, keys['itad'], batch)
print(stats)

In [ ]:
stats = itad_api.collect_price_history(conn, keys['itad'], shops=[61], country='PH')
print(stats)

## Stage 6 — SteamCharts historical CCU

Scrapes `steamcharts.com/app/{appid}/chart-data.json` for ~weekly historical concurrent-player counts going back to 2012. This is the primary input for the Part 2 sale-uplift analysis: for each ITAD sale event, we'll measure CCU in the sale window vs. a pre-sale baseline.

Throttled to 2s/req with a descriptive User-Agent. ~3h for 5,000 games. Uses `pending_appids_with_game` so only appids with a `games` row are attempted (FK requirement).

In [ ]:
# Small batch first — SteamCharts is throttled at 2s/req.
batch = db.pending_appids_with_game(conn, 'has_steamcharts', limit=20)
print(f'fetching CCU history for {len(batch)} games...')
stats = steamcharts.collect_history(conn, batch)
print(stats)

## Sanity checks

In [ ]:
summary = pd.read_sql('''
    SELECT
      (SELECT COUNT(*) FROM app_list)              AS app_list,
      (SELECT COUNT(*) FROM games)                 AS games,
      (SELECT COUNT(*) FROM game_genres)           AS genre_rows,
      (SELECT COUNT(*) FROM game_categories)       AS category_rows,
      (SELECT COUNT(*) FROM reviews_summary)       AS reviews_summary,
      (SELECT COUNT(*) FROM review_timestamps)     AS review_rows,
      (SELECT COUNT(*) FROM steamspy)              AS steamspy,
      (SELECT COUNT(*) FROM steamspy_tags)         AS steamspy_tags,
      (SELECT COUNT(*) FROM itad_mapping)          AS itad_mapping,
      (SELECT COUNT(*) FROM price_history)         AS price_history_rows,
      (SELECT COUNT(*) FROM steamcharts_history)   AS steamcharts_rows,
      (SELECT COUNT(*) FROM player_counts)         AS player_count_rows
''', conn)
summary.T.rename(columns={0: 'rows'})

In [ ]:
# Per-stage progress on the worklist
pd.read_sql('''
    SELECT
      SUM(has_details)        AS details_done,
      SUM(has_reviews)        AS reviews_done,
      SUM(has_steamspy)       AS steamspy_done,
      SUM(has_itad_id)        AS itad_id_done,
      SUM(has_price_history)  AS price_history_done,
      SUM(has_steamcharts)    AS steamcharts_done,
      COUNT(*)                AS total
    FROM app_list
''', conn)